In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import inspect
import openpyxl
from joblib import Parallel, delayed
import shutil
import gc
import logging
import sys
sys.path.insert(0, r"C:\Users\LAB-ADMIN\Documents\Actimotus2")

from analysis.activity_pipeline import load_raw_file, process_file


# Skru ned globalt
logging.basicConfig(level=logging.WARNING, force=True)

# Definer stier
file_path = Path(r"E:\Mobilize\Thigh_bl\Thigh_bl")
file_path_output = Path(r"E:\Mobilize\Thigh_bl\Thigh_bl\results")
file_path_output.mkdir(parents=True, exist_ok=True)

# Mappe til per-fil exposures CSV
exposures_dir = file_path_output / "exposures_csv"
exposures_dir.mkdir(parents=True, exist_ok=True)

# i hbsc - skal akserne roteres sådan her: Efter nærmere eftertanke, så tror jeg det er forkert. Det burde være x=y, y=-x, z=z
# df['acc_y'] = df['z'] * (-1)
# df['acc_x'] = df['y'] 
# df['acc_z'] = df['x'] * (-1)




# Definér rotations-funktion
def rotate_axes(df):
    # rotere akserne, så de stemmer overens med standard
    df['acc_y'] = df['y'] *(-1)
    df['acc_z'] = df['z'] 
    df['acc_x'] = df['x'] *(-1)


# Gem rotationskoden som tekst (til metadata-sheet)
axis_rotation_code = inspect.getsource(rotate_axes)

# Tilføj efter axis_rotation_code definition
error_log = file_path_output / "error_log.txt"


  
all_cwa_files = list(file_path.glob("**/*.cwa"))

unique_cwa_files = {}
duplicate_names = []

for cwa_file in all_cwa_files:
    if cwa_file.name in unique_cwa_files:
        duplicate_names.append(cwa_file.name)
    else:
        unique_cwa_files[cwa_file.name] = cwa_file

cwa_files = list(unique_cwa_files.values())

print(f"Finder {len(all_cwa_files)} filer i alt")
print(f"Beholder {len(cwa_files)} unikke filnavne")

if duplicate_names:
    print("Dubletter fundet for disse filnavne:")
    print(sorted(set(duplicate_names)))    
        
    


all_done = []
for cwa_file in cwa_files:
    df = load_raw_file(cwa_file)
    rotate_axes(df)
    df_subset = df[['acc_x', 'acc_y', 'acc_z']].copy()
    del df; gc.collect()
    all_done.append(process_file(df_subset, cwa_file.stem, file_path_output, exposures_dir, axis_rotation_code, error_log))
    del df_subset; gc.collect()

# tilføj log:
successful = sum(1 for r in all_done if r and r[1] == "SUCCESS")
failed = sum(1 for r in all_done if r and r[1] == "FAILED")

print(f"\n{'='*60}")
print(f"✓ Successful: {successful}/{len(cwa_files)}")
print(f"✗ Failed: {failed}/{len(cwa_files)}")
if failed > 0:
    print(f"Se fejl i: {error_log}")




# --- Stream-merge alle per-fil CSV til én samlet CSV uden at loade i RAM ---
combined_csv = file_path_output / "all_exposures.csv"
csv_files = sorted(exposures_dir.glob("*.csv"))

with open(combined_csv, "w", encoding="utf-8", newline="") as out_f:
    first = True
    for f in csv_files:
        with open(f, "r", encoding="utf-8") as in_f:
            if first:
                shutil.copyfileobj(in_f, out_f)  # inkl. header
                first = False
            else:
                next(in_f)  # skip header
                shutil.copyfileobj(in_f, out_f)

print(f"\nFærdig! Samlet exposures gemt i: {combined_csv}")

Finder 121 filer i alt
Beholder 121 unikke filnavne
Reading file... Done! (5.04s)
Converting to dataframe... Done! (0.52s)
Quality control... Done! (2.32s)
Processing: 19681_0002185811


Reading file... Done! (5.11s)
Converting to dataframe... Done! (0.53s)
Quality control... Done! (2.35s)
Processing: 19681_0002286811


Reading file... Done! (5.37s)
Converting to dataframe... Done! (0.54s)
Quality control... Done! (2.22s)
Processing: 19681_0002337411


Reading file... Done! (5.68s)
Converting to dataframe... Done! (0.56s)
Quality control... Done! (2.33s)
Processing: 21348_0002557711


Reading file... Done! (5.55s)
Converting to dataframe... Done! (0.55s)
Quality control... Done! (2.24s)
Processing: 21583_0002067811


...

Reading file... Done! (4.99s)
Converting to dataframe... Done! (0.58s)
Quality control... Done! (2.33s)
Processing: 36354_0002246411
Reading file... Done! (5.03s)
Converting to dataframe... Done! (0.61s)
Quality control... Done! (2.34s)
Processing: 36354_0002357711


Reading file... Done! (5.05s)
Converting to dataframe... Done! (0.57s)
Quality control... Done! (2.42s)
Processing: 36354_0002446911
Reading file... Done! (5.10s)
Converting to dataframe... Done! (0.59s)
Quality control... Done! (2.38s)
Processing: 36354_0002506311


Reading file... Done! (5.13s)
Converting to dataframe... Done! (0.58s)
Quality control... Done! (2.39s)
Processing: 36354_0002517011
Reading file... Done! (4.98s)
Converting to dataframe... Done! (0.64s)
Quality control... Done! (2.58s)
Processing: 36392_0004097111
Reading file... Done! (5.22s)
Converting to dataframe... Done! (0.58s)
Quality control... Done! (2.47s)
Processing: 36392_0004116611
Reading file... Done! (5.24s)
Converting to dataframe... Done! (0.57s)
Quality control... Done! (2.57s)
Processing: 36392_0004168511
Reading file... Done! (5.24s)
Converting to dataframe... Done! (0.60s)
Quality control... Done! (2.55s)
Processing: 36392_0004258611
Reading file... Done! (5.23s)
Converting to dataframe... Done! (0.60s)
Quality control... Done! (2.45s)
Processing: 36436_0002116311
Reading file... Done! (5.17s)
Converting to dataframe... Done! (0.57s)
Quality control... Done! (2.52s)
Processing: 36436_0002196711


Reading file... Done! (5.17s)
Converting to dataframe... Done! (0.59s)
Quality control... Done! (2.65s)
Processing: 36436_0002238511
Reading file... Done! (5.06s)
Converting to dataframe... Done! (0.60s)
Quality control... Done! (2.40s)
Processing: 36497_0001026911
Reading file... Done! (4.97s)
Converting to dataframe... Done! (0.61s)
Quality control... Done! (2.30s)
Processing: 36497_0001086611
Reading file... Done! (5.00s)
Converting to dataframe... Done! (0.54s)
Quality control... Done! (2.36s)
Processing: 36497_0001118611
Reading file... Done! (5.03s)
Converting to dataframe... Done! (0.58s)
Quality control... Done! (2.37s)
Processing: 36497_0001297511
Reading file... Done! (5.18s)
Converting to dataframe... Done! (0.60s)
Quality control... Done! (2.38s)
Processing: 36497_0001347011
Reading file... Done! (5.28s)
Converting to dataframe... Done! (0.61s)
Quality control... Done! (2.58s)
Processing: 36502_0002025911
Reading file... Done! (5.23s)
Converting to dataframe... Done! (0.60s

Reading file... Done! (5.24s)
Converting to dataframe... Done! (0.59s)
Quality control... Done! (2.44s)
Processing: 36502_0002437811
Reading file... Done! (5.18s)
Converting to dataframe... Done! (0.58s)
Quality control... Done! (2.47s)
Processing: 36514_0002047211
Reading file... Done! (5.26s)
Converting to dataframe... Done! (0.62s)
Quality control... Done! (2.51s)
Processing: 36514_0002567111
Reading file... Done! (7.38s)
Converting to dataframe... Done! (0.58s)
Quality control... Done! (2.50s)
Processing: 36529_0001047511
Reading file... Done! (5.23s)
Converting to dataframe... Done! (0.61s)
Quality control... Done! (2.49s)
Processing: 36529_0001097711

✓ Successful: 121/121
✗ Failed: 0/121

Færdig! Samlet exposures gemt i: E:\Mobilize\Thigh_16week\Thigh_16week\results\all_exposures.csv


In [ ]:
korrekt  df['acc_y'] = df['z'] * (-1)    
    df['acc_x'] = df['y'] 
    df['acc_z'] = df['x'] * (-1)